# Análise de desempenho por provisionamento (POI)
Dados do ApacheBench (`resultados_consolidados.csv`) e do `top` (`recursos_consolidados.csv`).
O teste **io de 2c1g (n=10000)** sempre falhava (disco cheio na VM) e é descartado.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110
BASE = Path("reports")
OUT = BASE / "plots"; OUT.mkdir(exist_ok=True)

df = pd.read_csv(BASE / "resultados_consolidados.csv", sep=";", encoding="utf-8-sig")
rec = pd.read_csv(BASE / "recursos_consolidados.csv", sep=";", encoding="utf-8-sig")

# descarta o teste que sempre caia (2c1g, io, manual)
df = df[~((df.cenario == "2c1g") & (df.fonte == "manual_ab_n10000") & (df.endpoint == "io"))].copy()
df = df.dropna(subset=["req_por_s"])

df["cpus"] = df.cenario.str[0].astype(int)
df["mem_gb"] = df.cenario.str[2].astype(int)

# rótulos legíveis para os cenários (1c1g -> "1 CPU / 1 GB")
def rotulo(c):
    return f"{c[0]} CPU{'s' if c[0] != '1' else ''} / {c[2]} GB"
ordem = [rotulo(c) for c in ["1c1g", "1c2g", "2c1g", "2c2g", "4c4g"]]
df["cenario"] = pd.Categorical(df.cenario.map(rotulo), ordem, ordered=True)
rec["cenario"] = pd.Categorical(rec.cenario.map(rotulo), ordem, ordered=True)
df["endpoint"] = df.endpoint.map({"cpu": "CPU", "memory": "Memory", "io": "IO"})
df["memoria"] = df.mem_gb.astype(str) + " GB"

bat = df[df.fonte == "resultado_ab_n1000"]   # varredura de concorrência
man = df[df.fonte == "manual_ab_n10000"]     # carga longa, c=100
from matplotlib.ticker import FuncFormatter

def em_milhares(eixo):
    """Mostra os ticks divididos por 1000 (o rótulo do eixo deve indicar ×10³)."""
    eixo.set_major_formatter(FuncFormatter(lambda v, _: f"{v / 1000:g}"))

print(len(bat), "linhas na bateria;", len(man), "no teste manual")
df.head()

48 linhas na bateria; 14 no teste manual


,cenario,fonte,endpoint,concorrencia,requisicoes,falhas,tempo_total_s,req_por_s,tempo_medio_req_ms,tempo_medio_concorrente_ms,...,total_max_ms,p50_ms,p75_ms,p90_ms,p95_ms,p99_ms,observacao,cpus,mem_gb,memoria
0,1 CPU / 1 GB,resultado_ab_n1000,CPU,10,1000.0,0.0,111.011,9.01,1110.113,111.011,...,1297.0,1087.0,1103.0,1128.0,1142.0,1179.0,NaN,1,1,1 GB
1,1 CPU / 1 GB,resultado_ab_n1000,CPU,50,1000.0,0.0,110.827,9.02,5541.341,110.827,...,7490.0,5424.0,5601.0,6013.0,6230.0,6445.0,NaN,1,1,1 GB
2,1 CPU / 1 GB,resultado_ab_n1000,CPU,100,1000.0,0.0,113.099,8.84,11309.946,113.099,...,15088.0,11083.0,11292.0,11732.0,12562.0,13643.0,NaN,1,1,1 GB
3,1 CPU / 1 GB,resultado_ab_n1000,CPU,200,1000.0,0.0,114.084,8.77,22816.775,114.084,...,26157.0,22365.0,22675.0,23017.0,23339.0,25128.0,NaN,1,1,1 GB
4,1 CPU / 1 GB,resultado_ab_n1000,Memory,10,1000.0,0.0,20.113,49.72,201.132,20.113,...,349.0,178.0,188.0,201.0,213.0,319.0,NaN,1,1,1 GB


## 1. Vazão (req/s) na carga longa (n=10000, c=100)

In [2]:
fig, ax = plt.subplots(figsize=(9, 5.5))
sns.barplot(data=man, x="endpoint", y="req_por_s", hue="cenario", ax=ax)
ax.set(title="Requisições por segundo (n=10000, concorrência 100)", xlabel="Endpoint", ylabel="Requisições por segundo")
for c in ax.containers: ax.bar_label(c, fmt="%.1f", fontsize=7)
ax.legend(title="Configuração da VM", loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=5, frameon=False)
fig.tight_layout(); fig.savefig(OUT / "vazao_manual.png", bbox_inches="tight"); plt.show()

C:\Projetos\UFCG\POI\.venv\lib\site-packages\ipykernel_launcher.py:6: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  


## 2. Latência (média, p95, p99) na carga longa

In [3]:
nomes = {"total_media_ms": "Latência\nmédia", "p95_ms": "Percentil 95\n(p95)", "p99_ms": "Percentil 99\n(p99)"}
lat = man.melt(id_vars=["cenario", "endpoint"], value_vars=list(nomes),
               var_name="metrica", value_name="ms").dropna()
lat["metrica"] = lat.metrica.map(nomes)
g = sns.catplot(data=lat, x="metrica", y="ms", hue="cenario", col="endpoint", kind="bar",
                sharex=True, sharey=False, height=4.2, aspect=1.1, legend_out=False)
g.set_titles("{col_name}"); g.set_axis_labels("Métrica de latência", "Latência (×10³ ms)")
for ax in g.axes.flat: em_milhares(ax.yaxis)
g.figure.suptitle("Latência por métrica (concorrência 100)", y=1.03)
sns.move_legend(g, "upper center", bbox_to_anchor=(0.5, -0.03), ncol=5, title="Configuração da VM", frameon=False)
g.savefig(OUT / "latencia_manual.png", bbox_inches="tight"); plt.show()

C:\Projetos\UFCG\POI\.venv\lib\site-packages\ipykernel_launcher.py:11: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  # This is added back by InteractiveShellApp.init_path()


## 3. Efeito da concorrência (bateria n=1000)

In [4]:
bat = bat.assign(cenario=bat.cenario.cat.remove_unused_categories())
for y, rotulo_y, titulo, arq in [("req_por_s", "Requisições por segundo", "Vazão x concorrência", "vazao_vs_concorrencia.png"),
                                 ("p95_ms", "Latência p95 (×10³ ms)", "Latência p95 x concorrência", "p95_vs_concorrencia.png")]:
    g = sns.relplot(data=bat, x="concorrencia", y=y, hue="cenario", style="cenario",
                    col="endpoint", kind="line", marker="o", facet_kws={"sharey": False},
                    height=4, legend="full")
    g.set(xscale="log", xticks=[10, 50, 100, 200], xticklabels=[10, 50, 100, 200])
    if y == "p95_ms":
        for ax in g.axes.flat: em_milhares(ax.yaxis)
    g.set_titles("{col_name}"); g.set_axis_labels("Requisições concorrentes", rotulo_y)
    g.figure.suptitle(titulo, y=1.03)
    sns.move_legend(g, "upper center", bbox_to_anchor=(0.5, -0.02), ncol=4, title="Configuração da VM", frameon=False)
    g.savefig(OUT / arq, bbox_inches="tight"); plt.show()

C:\Projetos\UFCG\POI\.venv\lib\site-packages\ipykernel_launcher.py:13: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  del sys.path[0]
C:\Projetos\UFCG\POI\.venv\lib\site-packages\ipykernel_launcher.py:13: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  del sys.path[0]


## 4. Heatmap de vazão (bateria): cenário x concorrência

In [5]:
for ep in ["CPU", "Memory", "IO"]:
    p = bat[bat.endpoint == ep].pivot_table(index="cenario", columns="concorrencia",
                                            values="req_por_s", observed=True)
    fig, ax = plt.subplots(figsize=(6.5, 4))
    sns.heatmap(p, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax, cbar_kws={"label": "Requisições por segundo"})
    ax.set(title=f"Vazão no endpoint {ep}", xlabel="Requisições concorrentes", ylabel="Configuração da VM")
    fig.tight_layout(); fig.savefig(OUT / f"heatmap_vazao_{ep.lower()}.png", bbox_inches="tight"); plt.show()

C:\Projetos\UFCG\POI\.venv\lib\site-packages\ipykernel_launcher.py:7: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  import sys


## 5. Speedup em relação a 1c1g (carga longa) e efeito de CPU vs. memória

In [6]:
base = man[man.cenario == ordem[0]].set_index("endpoint").req_por_s
sp = man.assign(speedup=man.apply(lambda r: r.req_por_s / base[r.endpoint], axis=1))

cores_mem = {"1 GB": "#4C72B0", "2 GB": "#DD8452", "4 GB": "#55A868"}
desloc = {"1 GB": (11, -4, "left"), "2 GB": (-11, -4, "right"), "4 GB": (11, -4, "left")}   # evita rótulos sobrepostos
fig, axs = plt.subplots(1, 3, figsize=(12, 4.6), sharey=True)
for ax, ep in zip(axs, ["CPU", "Memory", "IO"]):
    ax.plot([1, 4], [1, 4], "k:", lw=1.2, label="Escala linear (ideal)")
    ax.axhline(1, color="gray", lw=0.8)
    for mem, g in sp[sp.endpoint == ep].groupby("memoria"):
        ax.scatter(g.cpus, g.speedup, s=90, color=cores_mem[mem], label=mem, zorder=3)
        for _, r in g.iterrows():
            dx, dy, ha = desloc[mem]
            if r.cpus == 1 and r.speedup < 1:   # abaixo da base: rótulo embaixo do ponto
                dx, dy, ha = 0, -19, "center"
            ax.annotate(f"{r.speedup:.2f}x", (r.cpus, r.speedup), textcoords="offset points", xytext=(dx, dy), ha=ha,
                       fontsize=11, fontweight="bold", color=cores_mem[mem])
    ax.set(title=ep, xlabel="Número de vCPUs", xticks=[1, 2, 4], xlim=(0.45, 5.0))
axs[0].set_ylabel("Speedup (vezes mais rápido)")

# legenda única: um item por memória + a linha ideal
handles = {}
for ax in axs:
    for h, l in zip(*ax.get_legend_handles_labels()): handles.setdefault(l, h)
nomes_leg = [n for n in ["1 GB", "2 GB", "4 GB", "Escala linear (ideal)"] if n in handles]
fig.legend([handles[n] for n in nomes_leg], nomes_leg, title="Memória da VM", loc="upper center",
           bbox_to_anchor=(0.5, 0.02), ncol=4, frameon=False)
fig.suptitle("Speedup em relação a 1 CPU / 1 GB (concorrência 100)", y=1.02)
fig.tight_layout(); fig.savefig(OUT / "speedup.png", bbox_inches="tight"); plt.show()

print("Média req/s por vCPUs / memória (carga longa):")
display(man.pivot_table(index=["endpoint", "cpus"], columns="mem_gb", values="req_por_s", observed=True).round(2))

Média req/s por vCPUs / memória (carga longa):


C:\Projetos\UFCG\POI\.venv\lib\site-packages\ipykernel_launcher.py:10: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  # Remove the CWD from sys.path while we load stuff.


mem_gb             1      2       4
endpoint cpus                      
CPU      1      8.43   8.13     NaN
         2     11.68  14.52     NaN
         4       NaN    NaN   22.67
IO       1     22.25  25.30     NaN
         2       NaN  11.35     NaN
         4       NaN    NaN   22.53
Memory   1     51.57  43.92     NaN
         2     71.05  84.49     NaN
         4       NaN    NaN  130.74

## 6. Uso de recursos (`top`, durante a bateria)

In [7]:
fig, axs = plt.subplots(1, 2, figsize=(12, 5.5))
r = rec.set_index("cenario")
cpu = r[["cpu_uso_medio_pct", "cpu_uso_medio_sob_carga_pct", "cpu_uso_max_pct"]]
cpu.columns = ["Uso médio de CPU (toda a bateria)", "Uso médio de CPU (apenas sob carga, >20%)", "Pico de uso de CPU"]
mem = r[["mem_usada_media_mib", "mem_usada_max_mib", "mem_total_mib"]]
mem.columns = ["Memória usada (média)", "Memória usada (pico)", "Memória total da VM"]
cpu.plot.bar(ax=axs[0], rot=0)
axs[0].set(title="CPU", xlabel="Configuração da VM", ylabel="Uso de CPU (%)")
mem.plot.bar(ax=axs[1], rot=0)
axs[1].set(title="Memória", xlabel="Configuração da VM", ylabel="Memória (×10³ MiB)")
em_milhares(axs[1].yaxis)
for ax in axs:
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=1, frameon=False)
fig.suptitle("Uso de CPU e memória da VM durante a bateria de testes (medido com top)", y=1.02)
fig.tight_layout(); fig.savefig(OUT / "recursos.png", bbox_inches="tight"); plt.show()
rec


C:\Projetos\UFCG\POI\.venv\lib\site-packages\ipykernel_launcher.py:15: UserWarning: Matplotlib is currently using agg, which is a non-GUI backend, so cannot show the figure.
  from ipykernel import kernelapp as app


,cenario,amostras,mem_total_mib,mem_usada_media_mib,mem_usada_max_mib,cpu_uso_medio_pct,cpu_uso_max_pct,cpu_uso_medio_sob_carga_pct,amostras_sob_carga
0,1 CPU / 1 GB,725,957.4,237.5,303.6,59.4,86.0,65.0,660
1,1 CPU / 2 GB,795,1963.9,267.3,366.8,54.4,87.1,64.8,664
2,2 CPUs / 1 GB,675,957.2,385.7,582.6,29.8,93.8,45.7,393
3,2 CPUs / 2 GB,645,1963.7,392.8,638.5,37.8,89.1,52.7,425


## 7. Tabela resumo

In [8]:
resumo = man.pivot_table(index="cenario", columns="endpoint",
                         values=["req_por_s", "total_media_ms", "p99_ms"], observed=True).round(1)
resumo.to_csv(BASE / "resumo_manual.csv", sep=";", encoding="utf-8-sig")
resumo

p99_ms                  req_por_s              total_media_ms  \
endpoint           CPU       IO  Memory       CPU    IO Memory            CPU   
cenario                                                                         
1 CPU / 1 GB   14188.0   8097.0  2460.0       8.4  22.2   51.6        11811.0   
1 CPU / 2 GB   16221.0   8419.0  3659.0       8.1  25.3   43.9        12236.0   
2 CPUs / 1 GB  16900.0      NaN  2210.0      11.7   NaN   71.0         8509.0   
2 CPUs / 2 GB  11605.0  16826.0  1960.0      14.5  11.4   84.5         6840.0   
4 CPUs / 4 GB   9232.0   8359.0  1571.0      22.7  22.5  130.7         4376.0   

                               
endpoint           IO  Memory  
cenario                        
1 CPU / 1 GB   4460.0  1913.0  
1 CPU / 2 GB   3918.0  2248.0  
2 CPUs / 1 GB     NaN  1383.0  
2 CPUs / 2 GB  8774.0  1160.0  
4 CPUs / 4 GB  4410.0   742.0